## 🚀 AIOS Block Scaling & Load Testing: Strategies, Policies, and Metrics

- Author: Shridhar Kini (Profile)
- To Securely Run: `jupyter notebook password` to generate onetime password for secure access
- To Run: `jupyter notebook --allow-root  --port 9999 --ip=0.0.0.0`
- To Clear Outputs: Use `jupyter nbconvert --clear-output --inplace ScaleTesting.ipynb`

This notebook provides an overview of the **Scaling/Load Testing of Blocks,** in the AIOS platform. These tutorial is designed to helps users to scale the block based on varying demand and monitor their metrics effectively using metrics system.

### 🏙️ **Scaling**
- Scaling is the process of adjusting the number of instances of a block to meet changing demand or workload requirements.
- Scaling can be done manually or automatically based on predefined policies and rules.
- Scaling of instance across the nodes of same cluster is supported in AIOS v1.
- Use AIOS v1 Block Policy concept to define the scaling policies. example
    - Resource Allocation Policy
    - Load Balancer Policy
    - Auto-scaling Policy
    - Health Check Policy

#### 🖥️ **Resource Allocation Policy** 
- [Access Code here](policies/resource_allocator)
- Defines where the block instances should be deployed within the cluster.
- Can include rules based on blocks resource requirement and nodes resource availability, etc.
- Example: 
    - Deploy instances on nodes with at least 4 CPU and 8GB RAM available.
    - Deploy instances on nodes with at least 1 GPU with 35GB RAM available.
    - Avoid nodes with less than 20% of vcpu available.
    - Deploy instances on nodes with 1 GPU with 35GB RAM available or 2GPU with 16GB RAM available in each.
- Read this Documentation for more details on Resource Allocation Policy and its parameters: [Resource Allocation Policy Documentation](docs/README_resource_allocation.ipynb)

#### ⚖️ **Load Balancer Policy** 
- [Access Code here](policies/weighted_metrics_load_balancer)
- Manages the distribution of incoming requests across the block instances.
- Can include rules for session persistence, request routing, etc.
- Example: 
    - Route all requests from a specific user to the same instance.
    - Route all requests from a specific user session to the same instance.
    - Distribute requests evenly across all instances.
    - Route requests based on the geographic location of the user.
    - Route requests based on the type of request

- Read this Documentation for more details on Load Balancer Policy and its parameters: [Load Balancer Policy Documentation](docs/README_loadbalancer.ipynb)

#### 📈 **Auto-scaling Policy** 
- [Access Code here](policies/queue_length_scaling_metrics)
- Defines the conditions under which the block instances should be scaled up or down.
- Can include rules based on resource utilization, request rates, pending tasks(i.e queue length) etc.
- Example: 
    - Scale up when average queue length > 100 for 5 minutes.
    - Scale down when average queue length < 20 for 10 minutes.
    - Dont scale down all instance at once, keep at least 1 instance running always.
- Read this Documentation for more details on Autoscaler Policy and its parameters: [Autoscaler Policy Documentation](docs/README_autoscaler.ipynb)



#### ❤️‍🩹 **Health Check Policy** 
- [Access Code here](policies/block_healthcheck_policy)
- Defines the health check mechanisms for the block instances.
- Can include rules for determining the health status of an instance.
- Can call reassignelment of instances to other nodes if the instance is marked as unhealthy.
- Example: 
    - Mark an instance as unhealthy if it fails to respond to health checks for 3 minutes.
- Paramters and Settings:
    "failure_threshold": number of failed health checks before marking instance as reassignment.
    "check_interval_sec": frequency of health checks in seconds,
    "GATEWAY_URL": Gateway URL to call reassignement from health check policy,

### 🧪 **Load Testing Strategy**
- Load testing is the process of simulating real-world usage scenarios to evaluate the performance and scalability of a block.
- We used custom scripts to generate load on the block and monitor its performance.
- The load testing strategy includes the following steps:
    - Understand the Models capabilities and limitations;
        - Evaluate the model size(Disk, RAM, CPU, GPU requirements), Tokens limits, Latency, Throughput, 
        - Example:
            - Model: mistralai/Magistral-Small-2506
            - data type: fp16
            - Parameters: 23.6B
            - Disk: 45-46GB
            - GPU RAM: 45-47GB
            - Max Tokens: 40960
            - Latency: 9-10Sec (for 256 tokens)
    - Understand Library used for running the inference:
        - Batching capability, Model Splitting capability (Like Tensor Parallelism, Pipeline Parallelism)etc.
            - Example: 
                - Model: mistralai/Magistral-Small-2506
                - Library: vLLM
                - Batching: Yes (currently in our test 4 batch size is used)
                - Model Splitting: Yes (Tensor Parallelism)
    - Define the load testing scenarios:
        - Crowd Behaviour:
            - Number of users: 100
        - User Behaviour:
            - Sessions per User: 2
            - Request rate per user session: check [config.yaml](config.yaml)
    - Understand HW avaialble wrt the model requirements:
        - Example:
            - Model: mistralai/Magistral-Small-2506
            - In our case Nodes Details are as follows:
                - Node 1: 1x A100 80GB GPU, 12 vCPU
                - Node 2: 2x A100 160GB(80GB each) GPU, 24 vCPU
                - Node 4: 4x L4 92GB GPU(23GB each), 48 vCPU
                - Node 7: 2x A100 160GB GPU(80GB Each), 24 vCPU
                - Node 9: 4x A100 320GB GPU(80GB Each),48 vCPU
            - Calculate Max instances that can be run:(based on GPU RAM)
                - One instance takes 47GB GPU RAM, So only one instance can be run on A100 80GB GPU
                - Three instances can be run on 2xA100 160GB GPU
                    - One One instance can be run on each A100 GPU:
                    - One more instance with Tensor Parallelism(TP) across two A100 GPUs
                - One instance can be run on 4xL4 92GB GPU with TP of 4
                - Throughput Descrepencies: 
                    - for model mistralai/Magistral-Small-2506
                        - With 1 instance on A100: 30-31 tokens/sec generated with latency of 14-15 sec for 256 tokens when batched with 4 requests
                            - i.e effectvely 60-62 tokens/sec from 2 GPU's X 4 = 240-248 tokens/sec
                        - With 3 instances on 2xA100: 
                            - 14-15 tokens/sec per instance when 3 instances are run 
                                - i.e effectively 42-45 tokens/sec from 2 GPU's X 4 = 168-180 tokens/sec
                        - Why tokens/sec reduced:
                            - GPU utility is 100% when multiple instances are run
                        - Images below shows the GPU utlization when
                            ##### When 1 instance per GPU
                            - ![When 1 instance per GPU](screenshots/1.png)

                            ##### When 3 instances on 2xA100
                            - ![When 3 instances on 2xA100](screenshots/2.png)

                            ##### When 1 instance on 4xL4
                            - ![When 6 instances on 4xA100](screenshots/3.png)

            - Total instances that can be run on the cluster: 14
                - 1 on Node1
                - 3 on Node2
                - 1 on Node4
                - 3 on Node7
                - 6 on Node9

            - Effective Throughput that can be achieved on the cluster:
                - 1 on Node1: 1x4x30-31 tokens/sec = 120-124 tokens/sec
                - 3 on Node2: 3x4x14-15 tokens/sec = 168-180 tokens/sec
                - 1 on Node4: 1x4x18-19 tokens/sec = 72-76 tokens/sec
                - 3 on Node7: 3x4x14-15 tokens/sec = 168-180 tokens/sec
                - 6 on Node9: 6x4x14-15 tokens/sec = 336-360 tokens/sec
                - Total: 864-920 tokens/sec (approx)
            - tokens per request used in benchmark: 512:
                - 864-920/512 = 1.68-1.80 requests/sec
                - In our test we will be using 2 requests/sec for 100 users with 2 sessions each
                    i.e 2 requests/sec X 3600 sec = 7200 requests per hour
                    - We will be testing with this number for our load test
    - Logging/Monitoring:
        - Log the request and response times, errors, and other relevant metrics to DB.
            - We will be using TimescaleDB to storing the metrics.
            - Custom scripts to visualize above metrics
        - Monitor the resource utilization and metrics of the block instances.
            - We will be using Prometheus and Grafana for monitoring and visualization.
    - Monitor the block performance during the load test
        - check for latency, error rates, queue lengths, scaling up/down, total request processed etc.


**To  GET PORT MAPPING wrt to Service**([Doc](https://docs.aigr.id/installation/installation/#deploying-registry-services))

In [ ]:
# 🔧 Configuration Setup - Run this cell first to set up shared variables
import os

# Set configuration variables that will be available across all cells
GATEWAY_URL = "MANAGEMENTMASTER:30600"
CLUSTER_ID = "gcp-cluster-2"
GLOBAL_CLUSTER_METRICS_DB = "MANAGEMENTMASTER:30202"
GLOBAL_BLOCK_METRICS_DB = "MANAGEMENTMASTER:30201"
PARSER_URL = "MANAGEMENTMASTER:30501"
GLOBAL_CLUSTER_DB = "MANAGEMENTMASTER:30101"
GLOBAL_TASK_DB_SERVICE = "MANAGEMENTMASTER:30108"
COMPONENT_REGISTRY_SERVICE = "MANAGEMENTMASTER:30112"
GLOBAL_BLOCKDB_SERVICE = "MANAGEMENTMASTER:30100"
#SERVER_URL = "10.10.10.10:5000"  # For other API calls

# Set environment variables for bash cells
os.environ['GATEWAY_URL'] = GATEWAY_URL
os.environ['CLUSTER_ID'] = CLUSTER_ID
os.environ['GLOBAL_CLUSTER_METRICS_DB'] = GLOBAL_CLUSTER_METRICS_DB
os.environ['GLOBAL_BLOCK_METRICS_DB'] = GLOBAL_BLOCK_METRICS_DB
os.environ['PARSER_URL'] = PARSER_URL
os.environ['GLOBAL_CLUSTER_DB'] = GLOBAL_CLUSTER_DB
os.environ['GLOBAL_TASK_DB_SERVICE'] = GLOBAL_TASK_DB_SERVICE
os.environ['COMPONENT_REGISTRY_SERVICE'] = COMPONENT_REGISTRY_SERVICE
os.environ['GLOBAL_BLOCKDB_SERVICE'] = GLOBAL_BLOCKDB_SERVICE
#os.environ['SERVER_URL'] = SERVER_URL

print("✅ Configuration variables set:")
print(f"   • GATEWAY_URL: {GATEWAY_URL}")
print(f"   • CLUSTER_ID: {CLUSTER_ID}")
print(f"   • GLOBAL_CLUSTER_METRICS_DB: {GLOBAL_CLUSTER_METRICS_DB}")
print("\n📝 These variables are now available in both Python and bash cells!")
print("   - In Python: use GATEWAY_URL, CLUSTER_ID, GLOBAL_CLUSTER_METRICS_DB PARSER_URL GLOBAL_CLUSTER_DB GLOBAL_TASK_DB_SERVICE")
print("   - In bash: use $GATEWAY_URL, $CLUSTER_ID, $GLOBAL_CLUSTER_METRICS_DB $PARSER_URL $GLOBAL_CLUSTER_DB $GLOBAL_TASK_DB_SERVICE")
os.system('echo $GATEWAY_URL')
os.system('echo $CLUSTER_ID')
os.system('echo $GLOBAL_CLUSTER_METRICS_DB')
os.system('echo $PARSER_URL')
os.system('echo $GLOBAL_CLUSTER_DB')
os.system('echo $GLOBAL_TASK_DB_SERVICE')
os.system('echo $COMPONENT_REGISTRY_SERVICE')
os.system('echo $GLOBAL_BLOCKDB_SERVICE')
os.system('echo $GLOBAL_BLOCK_METRICS_DB')

### **Resource Allocation Policy** [Code](policies/resource_allocator)

##### Register the component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/registerComponent \
 -d @./policies/resource_allocator/registration.json \
 -H "Content-Type: application/json" | json_pp

##### Unregister Component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.scaletest_block_resource_allocator:1.0.0-stable"}' | json_pp

##### Upload the Zip file of the policy

In [ ]:
%%bash
bash policies/resource_allocator/upload.sh

### **Load Balancer Policy** [Code](policies/weighted_metrics_load_balancer)

##### Register the component

In [ ]:
%%bash
bash policies/weighted_metrics_load_balancer/register.sh

##### Unregister Component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.weightedmetricsloadbalancer:2.0-stable"}' | json_pp

##### Upload the Zip file of the policy

In [ ]:
%%bash
bash policies/weighted_metrics_load_balancer/upload.sh

### **Auto-scaling Policy** [Code](policies/queue_length_scaling_metrics)

##### Register the component

In [ ]:
%%bash
bash policies/queue_length_scaling_metrics/register.sh

##### Unregister Component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.queuebasedautoscaler:2.0-stable"}' | json_pp

##### Upload the Zip file of the policy

In [ ]:
%%bash
bash policies/queue_length_scaling_metrics/upload.sh

### **Health Check Policy** [Code](policies/block_healthcheck_policy)

##### Register the component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/registerComponent \
 -d @./policies/block_healthcheck_policy/block_health_check_registration.json \
 -H "Content-Type: application/json" | json_pp

##### Unregister Component

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.demo_block_health_checker:1.0.0-stable"}' | json_pp

##### Upload the Zip file of the policy

In [ ]:
%%bash
bash policies/block_healthcheck_policy/upload.sh

### **Create Blocks in Cluster(With Above Policies)** [Code](model_code)

#### **Build Docker Image**

In [ ]:
%%bash
bash model_code/build_docker_vllm.sh

#### **Push Docker Image to Registry**

In [ ]:
%%bash
docker push MANAGEMENTMASTER:31280/vllm_batching_aios:v1

#### **Register Components**

In [ ]:
%%bash 
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/registerComponent \
  -H "Content-Type: application/json" \
  -d @./blocks/mistrall_vllm/component.json | json_pp

#### **Unregister Components**

In [ ]:
%%bash
curl -X POST http://$COMPONENT_REGISTRY_SERVICE/api/unregisterComponent \
  -H "Content-Type: application/json" \
  -d '{"uri":"model.magistral-small-2506-vllm:1.0.0-stable"}' | json_pp

#### **Deploy Block**

In [ ]:
%%bash
curl -X POST http://$PARSER_URL/api/createBlock \
 -H "Content-Type: application/json" \
 -d @./blocks/mistral_vllm/allocation.json | json_pp

#### **Get Block Details**

In [ ]:
%%bash
curl -X GET http://$GLOBAL_BLOCKDB_SERVICE/blocks/magistral-small-2506-vllm-block \
    -H "Content-Type: application/json" | json_pp

#### **Create Inference Server For The Cluster**
- Run this commands in your cluster node(like master node)

- `kubectl create namespace inference-server`
- `kubectl create -f inference_server/inference_server.yaml`

#### **Do the Inference**

In [ ]:
%%bash
curl -X POST  http://$GCP_CLUSTER_2_INFERENCE_SERVER:31504/v1/infer \
  -H "Content-Type: application/json" \
  -d '{
  "model": "magistral-small-2506-vllm-block",
  "session_id": "session-2",
  "seq_no": 16,
  "data": {
    "mode": "chat",
    "gen_params": {
      "temperature": 0.1,
      "top_p": 0.95,
      "max_tokens": 4096
    },
    "message": "Give me code for adding two integers list element wise in python",
    "system_message": "You are a helpful assistant that provides code examples."
  },
  "graph": {},
  "selection_query": {
    
  }
}'

#### **To Get Blocks Metrics**

In [ ]:
%%bash
curl -X GET http://$GLOBAL_BLOCK_METRICS_DB/block/magistral-small-2506-vllm-block | json_pp

#### **Remove Block**

In [ ]:
%%bash
curl -X POST http://$GATEWAY_URL/controller/removeBlock/gcp-cluster-2 \
    -H "Content-Type: application/json" \
    -d '{"block_id": "magistral-small-2506-vllm-block"}'

### 📊 **Run Response Time and Failures Visualization** [Code](streamlit_visualize_response_time.py)

- `bash setup_timescaledb_env.sh`
    - To Setup all env variables required to connect to TimescaleDB

In [ ]:
# Run this in screen, So that this UI is always available 
bash run_visualize_response_time.sh

- to get streamlit UI:
    - In browser : http://IP:8502/   
        - Port and IP will vary based on your setup
- Use the test_id generated from the `crowd.py` script to visualize the response time and failures. Filter the data for selected timeranges as shown below.

    ##### Average Response Time Graph (averaged over 60 seconds)
    ![Average Response Time Graph (averaged over 60 seconds)](screenshots/Streamlit_11.png)

    ##### Total Request Count and Failed request in first attempt
    ![Total Request Count](screenshots/Streamlit_12.png)

### 📉 **Create Grafana Dashboard for Blocks metrics** [Dashboard Json](Dashboard.json)

- Add new Data Source in Grafana
    - Name: `Prometheus`
    - Prometheus Server URL: `http://prometheus-server.metrics-storage.svc.cluster.local`
    - Open http://MANAGEMENTMASTER:32199/connections/datasources
- Open the Grafana Dashboard using the URL: http://MANAGEMENTMASTER:32199/dashboards
- Create a new dashboard and import the above json file.
    - Press `New` -> `Import` -> Upload the above json file -> Press `Import` button
- Now open the dashboard to see the metrics.
    - If any time block ID changes, update the dashboard json with new block ID or edit the panels to use the new block ID.

- Attached Fe Screnshots of the dashboard below:
    - ![Dashboard1](screenshots/gafana_1.png)
    - ![Dashboard2](screenshots/gafana_2.png)
    - ![Dashboard3](screenshots/gafana_3.png)
    - ![Dashboard4](screenshots/gafana_4.png)

### 👥 **Run Crowd Simulation Code**

- crowd.py file contains the code to simulate the crowd behaviour.
- To fetch random question, use script `generate_questions.py` and uses GEMINI API to fetch random questions. You may have to supply the API key in the script or ENV variable.
    - We have already generated 4.9k+ questions and stored in `questions.jsonl` file.
    - each session_ids will pick the questions from here, once all questions are over, again it starts taking questions from this file randomly. Randomness is added so that same question is not picked up by multiple sessions at the same time, such they they end up in same instance.
- install dependencies:
    - `pip install -r requirements.txt`
    - `pip install -r requirements_streamlit.txt`
    - `bash setup_timescaledb_env.sh`
        - To Setup all env variables required to connect to TimescaleDB
- To run the code, use the command: `python3 crowd.py`
- this code is responsible for generating the load on the block by simulating multiple users.
- better to run this code in screen  as this will run for longer duration configured in config.yaml file.
- user.py file contains the code for simulating a user with multiple sessions and request rates.
    - Upon each request it calls the block endpoint and logs the response time and failure time to DMA Endpoint DB.
        -   ```json{
            "block_id":  session.block_id,
            "session_id": session.session_id,
            "seq_no": seq_no,
            "type": "success",
            "response_time": 0.0,
            "raw": "{}",
            "test_id": self.test_id,
            "user_id": self.user_id,
            "starttime": time.time(),
            "endtime": time.time(),
            "starttimeObj": datetime.now(self.ist_tz),
            "endtimeObj": datetime.now(self.ist_tz)
        }```
    - It Divides Users request with equidistant time interval based on the request rate configured in config.yaml file. 
        - i.e 36 requests per hour means one request every 100 seconds. 
        - Since 2 Session, So 2 requests every 100 seconds.